# 08 — V3 Stage-4 calibration-limitation report

This notebook assembles only evidence licensed by the failed gate chain. It
publishes a calibration/READ-positive-control limitation and explicitly leaves
P1-P3 untested. The v1 correlation comparison is descriptive legacy evidence
from an invalidated instrument.

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = Path('/home/jovyan/j-space-thoughts')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

metrics = json.loads((ROOT / 'results/metrics.json').read_text())
v3 = metrics['calibration_v3']
assert v3['gate_ledger']['g_alpha'] == 'FAIL'
assert v3['selected_intervention'] is None
required = {
    '05_science_twohop.ipynb',
    '06_science_ambiguity.ipynb',
    '07_scale.ipynb',
}
assert set(v3['stage3_notebooks']) == required
assert all(
    row['status'] == 'SKIPPED_PREREQUISITE'
    for row in v3['stage3_notebooks'].values()
)
print('Gate chain verified: Stage 4 fallback is required.')

Gate chain verified: Stage 4 fallback is required.


In [2]:
from src.v3_fallback import record_stage4_fallback

metrics = record_stage4_fallback()
v3 = metrics['calibration_v3']
stage4 = v3['stage4_fallback']
assert stage4['status'] == 'COMPLETE'
assert stage4['claim_boundary']['hypothesis_status'] == 'NOT_TESTED'
assert stage4['claim_boundary']['hypothesis_false_established'] is False
assert v3['gate_ledger']['stage4_report'] == 'PASS'
print(json.dumps({
    'classification': stage4['classification'],
    'failed_gate': stage4['failed_gate'],
    'predictions': stage4['predictions'],
    'claim_boundary': stage4['claim_boundary'],
    'raw_artifact': stage4['raw_artifact'],
}, indent=2))

{
  "classification": "CALIBRATION_READ_POSITIVE_CONTROL_LIMITATION",
  "failed_gate": "G-ALPHA",
  "predictions": {
    "P1": "NOT_TESTED",
    "P2": "NOT_TESTED",
    "P3": "NOT_TESTED"
  },
  "claim_boundary": {
    "hypothesis_status": "NOT_TESTED",
    "hypothesis_false_established": false,
    "allowed_claim": "No frozen intervention strength passed G-ALPHA; the current intervention/weight-READ positive-control pair was not fully calibrated on Qwen2.5-7B.",
    "forbidden_claim": "The result must not be described as showing that the Written-vs-Read hypothesis is false."
  },
  "raw_artifact": {
    "path": "data/raw/v3/015_alpha_sweep.json",
    "bytes": 17102855,
    "sha256": "ef53591af7e331607d7fafc43d6f3d08e2687e68197b9e925f7ee1a60e0e0cd5"
  }
}


In [3]:
report = (ROOT / 'results/RESULTS.md').read_text()
required_phrases = [
    'V3 COMPLETE',
    'CALIBRATION_READ_POSITIVE_CONTROL_LIMITATION',
    'P1, P2, and P3 are **NOT TESTED**',
    'does **not** show that the Written-vs-Read hypothesis is false',
    '24/24',
    'N=155',
]
assert all(phrase in report for phrase in required_phrases)
print(report)
print('Notebook 08 complete. V3 report persisted without a hypothesis verdict.')

# Surgical intervention calibration report (v3)

## Current verdict

**V3 COMPLETE — CALIBRATION/READ-POSITIVE-CONTROL LIMITATION; NO HYPOTHESIS VERDICT.** G-SWAP passed, but no frozen alpha satisfied G-ALPHA. Stage 2 and Stage 3 were skipped by prerequisite, and P1-P3 remain untested.

## Environment

- GPU: NVIDIA H200; 143771 MiB total; 143072 MiB free at preflight.
- Home/HF-cache filesystem: 100.0 GiB total; 38.1 GiB free.
- Required tool/auth preflight: **PASS**.
- Model: `Qwen/Qwen2.5-7B-Instruct` at `a09a35458c702b33eeacc393d103063234e8bc28` in `torch.bfloat16`.

## Stage 0 — v2 instrument re-verification

- HF/J-Lens logit gate: **PASS**; max mean KL=1.660e-08, N=20.
- Known-answer alpha-2 swaps: **PASS** (3/3).
- Cached held-out G-DIR artifact: **PASS**; retrieval top-1=0.550, known-answer top-5=0.8875.
- Non-structural direct suppression controls: **PASS**.

Stage-0 decision: **PASS**. This licenses only G-SWAP confirmation
and the alpha sweep; it does not license Stage-2 re